In [0]:
import re
from pyspark.sql import Row

# Set your folder path (note: trailing slash optional)
folder_path = '/Volumes/xliidw_dev_lpl/application/adhoc_files/Staging_Genius_Analysis/'

# List all SQL/Text files in the folder (supports .sql and .txt)
files = [f for f in dbutils.fs.ls(folder_path) if f.name.lower().endswith(('.sql', '.txt'))]

print(f"Found {len(files)} SQL/TXT files\n")

results = []

for file in files:
    # Read file content - handle various encodings (UTF-16 BOM, UTF-8, latin1)
    local_path = file.path.replace("dbfs:", "")
    with open(local_path, "rb") as f:
        raw = f.read()
    if raw[:2] in (b'\xff\xfe', b'\xfe\xff'):
        sql_text = raw.decode('utf-16')
    else:
        try:
            sql_text = raw.decode('utf-8-sig')
        except UnicodeDecodeError:
            sql_text = raw.decode('latin1')

    # 1) Try to extract the stored procedure name from content
    m = re.search(r"CREATE\s+(?:ALTER\s+)?PROCEDURE\s+\[dbo\]\.\[(.*?)\]", sql_text, re.IGNORECASE | re.DOTALL)
    proc_name = m.group(1).strip() if m else None

    # 2) Fallback: derive from filename if content does not contain name
    if not proc_name:
        # Remove extension and any path-like parts
        proc_name = file.name
        proc_name = re.sub(r"\.(sql|txt)$", "", proc_name, flags=re.IGNORECASE)

    # 3) Extract table names after INSERT INTO
    insert_matches = re.findall(r'INSERT\s+INTO\s+([\w\.\[\]]+)', sql_text, re.IGNORECASE)

    # Clean brackets and take the last part after dot
    clean_tables = []
    for t in insert_matches:
        clean = re.sub(r'[\[\]]', '', t)  # remove brackets
        clean = clean.split('.')[-1]     # take last part after dot
        if clean:
            clean_tables.append(clean)

    insert_tables = sorted(set(clean_tables))

    results.append({
        "file": file.name,
        "proc_name": proc_name,
        "insert_tables": insert_tables
    })

    print(f"File: {file.name}")
    print(f"Proc Name: {proc_name}")
    print(f"Insert Tables: {insert_tables}")
    print("-" * 50)

# Save as Spark DataFrame for further use
results_df = spark.createDataFrame([Row(**r) for r in results])
display(results_df)

Found 8 SQL/TXT files

File: dbo.sp_AddDataCompression.StoredProcedure.sql
Proc Name: sp_AddDataCompression
Insert Tables: []
--------------------------------------------------
File: dbo.sp_AddDataCompression_MT_EP.StoredProcedure.sql
Proc Name: sp_AddDataCompression_MT_EP
Insert Tables: []
--------------------------------------------------
File: dbo.sp_AddDataCompression_RITM1159464_20201008.StoredProcedure.sql
Proc Name: sp_AddDataCompression_RITM1159464_20201008
Insert Tables: []
--------------------------------------------------
File: dbo.sp_Build_CorpClaimAmounts.StoredProcedure.sql
Proc Name: sp_Build_CorpClaimAmounts
Insert Tables: ['CorpClaimPayments', 'CorpReserveDeltaOutwards', 'CorpReserveDeltas']
--------------------------------------------------
File: dbo.sp_Build_CorpClaimAmounts_20072023.StoredProcedure.sql
Proc Name: sp_Build_CorpClaimAmounts_20072023
Insert Tables: ['CorpClaimPayments', 'CorpReserveDeltaOutwards', 'CorpReserveDeltas']
----------------------------------

file,proc_name,insert_tables
dbo.sp_AddDataCompression.StoredProcedure.sql,sp_AddDataCompression,List()
dbo.sp_AddDataCompression_MT_EP.StoredProcedure.sql,sp_AddDataCompression_MT_EP,List()
dbo.sp_AddDataCompression_RITM1159464_20201008.StoredProcedure.sql,sp_AddDataCompression_RITM1159464_20201008,List()
dbo.sp_Build_CorpClaimAmounts.StoredProcedure.sql,sp_Build_CorpClaimAmounts,"List(CorpClaimPayments, CorpReserveDeltaOutwards, CorpReserveDeltas)"
dbo.sp_Build_CorpClaimAmounts_20072023.StoredProcedure.sql,sp_Build_CorpClaimAmounts_20072023,"List(CorpClaimPayments, CorpReserveDeltaOutwards, CorpReserveDeltas)"
dbo.sp_Build_CorpClaimAmounts_Inc.StoredProcedure.sql,sp_Build_CorpClaimAmounts_Inc,"List(CorpClaimPayments, CorpReserveDeltaOutwards, CorpReserveDeltas)"
dbo.sp_Build_Incremental_Staging_DltIns.StoredProcedure.sql,sp_Build_Incremental_Staging_DltIns,List()
dbo.sp_WarehouseBuild.StoredProcedure.txt,sp_WarehouseBuild,List()
